### Building a Transformer-Based NMT Model

Keras provides some, but not all, of the building blocks that comprise an end-to-end
transformer. It provides a handy implementation of self-attention layers in its Multi
HeadAttention class, for example, but it doesn’t implement positional embedding.
However, a separate package named KerasNLP does. 

In [3]:
# Install Keras NLP 
!pip install keras-nlp

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 14.9 MB/s eta 0:00:00

   ---------------------------------------- 0/4 [kagglesdk]
   ---------------------------------------- 0/4 [kagglesdk]
   ---------------------------------------- 0/4 [kagglesdk]
   ---------------------------------------- 0/4 [kagglesdk]
   ---------------------------------------- 0/4 [kagglesdk]
   ---------------------------------------- 0/4 [kagglesdk]
   ---------- ----------------------------- 1/4 [kagglehub]
   ---------- ----------------------------- 1/4 [kagglehub]
   -------------------- ------------------- 2/4 [keras-hub]
   -------------------- ------------------- 2/4 [keras-hub]
   -------------------- ------------------- 2/4 [keras-hub]
   -------------------- ------------------- 2/4 [keras-hub]
   -------------------- ------------------- 2/4 [keras-hub]
   -------------------- ------------------- 2/4 [keras-hub]
   ----------

In [4]:
#Import libraries
import pandas as pd
import re
from unicodedata import normalize

In [3]:
#Import the dataset 
df=pd.read_csv("en-fr.txt",names=["en","fr","att"],usecols=['en', 'fr'], sep='\t')
df=df.sample(frac=1, random_state=42)
df = df.reset_index(drop=True)
df.head()

,en,fr
0,You're very clever.,Vous êtes fort ingénieuse.
1,Are there kids?,Y a-t-il des enfants ?
2,Come in.,Entrez !
3,Where's Boston?,Où est Boston ?
4,You see what I mean?,Vous voyez ce que je veux dire ?


The dataset needs to be cleaned before it’s used to train a model.

In [5]:
def clean_text(text):
    text=normalize('NFD',text.lower())
    text = re.sub('[^A-Za-z ]+', '', text)
    return text

def clean_and_prepare_text(text):
    text = '[start] ' + clean_text(text) + ' [end]'
    return text

df['en'] = df['en'].apply(lambda row: clean_text(row))
df['fr'] = df['fr'].apply(lambda row: clean_and_prepare_text(row))
df.head()

,en,fr
0,youre very clever,[start] vous etes fort ingenieuse [end]
1,are there kids,[start] y atil des enfants [end]
2,come in,[start] entrez [end]
3,wheres boston,[start] ou est boston [end]
4,you see what i mean,[start] vous voyez ce que je veux dire [end]


The next step is to scan the dataset and determine the maximum length of the English
phrases and of the French phrases. These lengths will determine the lengths of the
sequences input to and output from the model:

In [6]:
en=df['en']
fr=df['fr']
fr

0              [start] vous etes fort ingenieuse [end]
1                    [start] y atil des enfants  [end]
2                                 [start] entrez [end]
3                         [start] ou est boston  [end]
4        [start] vous voyez ce que je veux dire  [end]
                             ...                      
49995              [start] mon plan a fonctionne [end]
49996             [start] tom accepta le travail [end]
49997                [start] tom est fier de toi [end]
49998         [start] pouvonsnous nous en aller  [end]
49999                   [start] a chacun le sien [end]
Name: fr, Length: 50000, dtype: object

In [8]:
en_max_len=max(len(line.split()) for line in en)
fr_max_len=max(len(line.split()) for line in fr)
sequence_len=max(en_max_len, fr_max_len)

In [9]:
print(f'Max phrase length (English): {en_max_len}')
print(f'Max phrase length (French): {fr_max_len}')
print(f'Sequence length: {sequence_len}')

Max phrase length (English): 7
Max phrase length (French): 16
Sequence length: 16


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
en_tokenizer=Tokenizer()
en_tokenizer.fit_on_texts(en)
en_sequences=en_tokenizer.texts_to_sequences(en)
en_x=pad_sequences(en_sequences,maxlen=sequence_len, padding="post")